### Read Bronze data

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "olist_mlops.bronze.order_items"
SILVER_TABLE = "olist_mlops.silver.order_items_clean"

bronze_items = spark.table(BRONZE_TABLE)

print("Bronze rows:", bronze_items.count())
bronze_items.printSchema()

display(bronze_items.limit(5))

### Validate source values before casting

In [0]:
type_mapping = {
    "order_item_id": "INT",
    "shipping_limit_date": "TIMESTAMP_NTZ",
    "price": "DECIMAL(18,2)",
    "freight_value": "DECIMAL(18,2)"
}

for column, target_type in type_mapping.items():

    failed_casts = bronze_items.filter(
        F.col(column).isNotNull()
        & F.expr(
            f"try_cast(`{column}` AS {target_type})"
        ).isNull()
    ).count()

    print(f"{column}: {failed_casts} failed casts")

    assert failed_casts == 0, (
        f"Conversion errors in {column}"
    )

In [0]:
for column in ["price", "freight_value"]:

    invalid_precision = bronze_items.filter(
        F.col(column).isNotNull()
        & ~F.trim(F.col(column)).rlike(
            r"^-?[0-9]+(\.[0-9]{1,2})?$"
        )
    ).count()

    print(
        f"{column}: {invalid_precision} "
        "values with unsupported decimal format"
    )

    assert invalid_precision == 0

print("SOURCE DATA TYPE VALIDATION PASSED")

### Apply the Silver schema

In [0]:
silver_items = (
    bronze_items

    .withColumn(
        "order_item_id",
        F.expr(
            "try_cast(order_item_id AS INT)"
        )
    )

    .withColumn(
        "shipping_limit_date",
        F.expr(
            "try_cast(shipping_limit_date AS TIMESTAMP_NTZ)"
        )
    )

    .withColumn(
        "price",
        F.expr(
            "try_cast(price AS DECIMAL(18,2))"
        )
    )

    .withColumn(
        "freight_value",
        F.expr(
            "try_cast(freight_value AS DECIMAL(18,2))"
        )
    )

    .withColumn(
        "item_total_value",
        F.col("price") + F.col("freight_value")
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

silver_items.printSchema()

### Validate keys and row counts

In [0]:
source_count = bronze_items.count()
silver_count = silver_items.count()

null_keys = silver_items.filter(
    F.col("order_id").isNull()
    | F.col("order_item_id").isNull()
).count()

duplicate_keys = (
    silver_items
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_item_numbers = silver_items.filter(
    F.col("order_item_id") <= 0
).count()

print("Bronze rows:", source_count)
print("Silver rows:", silver_count)
print("Null composite keys:", null_keys)
print("Duplicate composite keys:", duplicate_keys)
print("Invalid item sequence numbers:", invalid_item_numbers)

assert source_count == silver_count == 112650
assert null_keys == 0
assert duplicate_keys == 0
assert invalid_item_numbers == 0

print("ORDER ITEMS KEY VALIDATION PASSED")

### Check missing, negative and zero values

In [0]:
financial_quality = silver_items.agg(

    F.count(
        F.when(F.col("price").isNull(), 1)
    ).alias("missing_price"),

    F.count(
        F.when(F.col("freight_value").isNull(), 1)
    ).alias("missing_freight"),

    F.count(
        F.when(F.col("price") < 0, 1)
    ).alias("negative_price"),

    F.count(
        F.when(F.col("freight_value") < 0, 1)
    ).alias("negative_freight"),

    F.count(
        F.when(F.col("freight_value") == 0, 1)
    ).alias("zero_freight"),

    F.min("price").alias("min_price"),
    F.max("price").alias("max_price")

)

display(financial_quality)

result = financial_quality.first()

assert result["missing_price"] == 0
assert result["missing_freight"] == 0
assert result["negative_price"] == 0
assert result["negative_freight"] == 0

print("FINANCIAL DATA VALIDATION PASSED")

### Validate foreign-key relationships in Databricks - Referential integrity checks

In [0]:
relationships = [
    (
        "Orders",
        "order_id",
        "olist_mlops.silver.orders_clean"
    ),
    (
        "Products",
        "product_id",
        "olist_mlops.silver.products_clean"
    ),
    (
        "Sellers",
        "seller_id",
        "olist_mlops.silver.sellers_clean"
    )
]

for name, key_column, parent_table in relationships:

    parent_keys = (
        spark.table(parent_table)
        .select(key_column)
        .distinct()
    )

    orphan_count = (
        silver_items
        .select(key_column)
        .distinct()
        .join(
            parent_keys,
            on=key_column,
            how="left_anti"
        )
        .count()
    )

    print(f"{name} orphan keys: {orphan_count}")

    assert orphan_count == 0

print("REFERENTIAL INTEGRITY VALIDATION PASSED")

### Write the Silver Order Items table - Save as Delta

In [0]:
(
    silver_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

written_count = spark.table(SILVER_TABLE).count()

assert written_count == 112650

print("Created:", SILVER_TABLE)
print("Rows:", written_count)

### Analyze order items using SQL - Overall item statistics

In [0]:
%sql

SELECT
    COUNT(*) AS total_items,
    COUNT(DISTINCT order_id) AS distinct_orders,
    COUNT(DISTINCT product_id) AS distinct_products,
    COUNT(DISTINCT seller_id) AS distinct_sellers,
    SUM(price) AS total_item_value,
    SUM(freight_value) AS total_freight_value,
    SUM(item_total_value) AS combined_value
FROM olist_mlops.silver.order_items_clean;

### Query B — Orders containing multiple items

In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS number_of_items,
    COUNT(DISTINCT seller_id) AS number_of_sellers,
    SUM(price) AS order_item_value,
    SUM(freight_value) AS freight_total
FROM olist_mlops.silver.order_items_clean
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY number_of_items DESC
LIMIT 10;

### Query C — Shipping deadline analysis

In [0]:
%sql

SELECT
    COUNT(*) AS total_items,
    MIN(shipping_limit_date) AS earliest_shipping_limit,
    MAX(shipping_limit_date) AS latest_shipping_limit,
    COUNT_IF(shipping_limit_date IS NULL) AS missing_shipping_limits
FROM olist_mlops.silver.order_items_clean;

In [0]:
%sql

SELECT
    YEAR(shipping_limit_date) AS shipping_year,
    COUNT(*) AS item_count
FROM olist_mlops.silver.order_items_clean
GROUP BY YEAR(shipping_limit_date)
ORDER BY shipping_year;